In [16]:
# Install new dependencies for Pipeline 2
!pip install chromadb sentence-transformers langchain-chroma langchain-huggingface tqdm pandas IProgress -q

In [17]:
# ============================================================
# PIPELINE 2: AI REVIEW CLASSIFIER + VECTOR DB
# ============================================================
import re
import operator
import pandas as pd
from typing import TypedDict, List
from tqdm.auto import tqdm
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langgraph.graph import START, END, StateGraph

print("All imports loaded.")

All imports loaded.


In [18]:
PIPELINE_CONFIG = {
    "csv_path": r"D:\New folder (6)\reviews_dataset.csv.gz",
    "chunksize": 5000,
    "chroma_persist_dir": "./chroma_reviews_db",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embed_batch_size": 5000,
}

AI_KEYWORDS = re.compile(
    r"\b(ai|artificial intelligence|machine learning|chatbot|"
    r"smart reply|autocomplete|ai[\s\-]generated|llm|gpt|gemini|"
    r"copilot|neural network|recommendation engine|voice assistant|"
    r"nlp|natural language|predictive|intelligent suggestion|"
    r"ai assistant|bard|claude|language model|deep learning)\b",
    re.IGNORECASE
)

print(f'Config loaded. CSV: {PIPELINE_CONFIG["csv_path"]}')

Config loaded. CSV: D:\New folder (6)\reviews_dataset.csv.gz


In [19]:
# --- PIPELINE 2 STATE ---
class ReviewPipelineState(TypedDict):
    csv_path: str
    persist_dir: str
    ai_reviews: List[dict]
    total_scanned: int
    keyword_hits: int
    vectorstore_ready: bool
    run_summary: str


# --- PIPELINE 2 NODES ---
def stream_and_filter(state: ReviewPipelineState) -> dict:
    """Stream the gzipped CSV in chunks and keep only AI-related reviews."""
    csv_path = state["csv_path"]
    matches = []
    total = 0

    reader = pd.read_csv(
        csv_path,
        chunksize=PIPELINE_CONFIG["chunksize"],
        compression="gzip",
        dtype=str,
        on_bad_lines="skip",
    )

    for chunk in tqdm(reader, desc="Scanning reviews", unit="chunk"):
        chunk = chunk.dropna(subset=["message"])
        total += len(chunk)
        mask = chunk["message"].apply(lambda msg: bool(AI_KEYWORDS.search(msg)))
        matched = chunk[mask]
        if not matched.empty:
            matches.extend(matched.to_dict(orient="records"))

    print(f'Scanned {total:,} rows — {len(matches):,} AI-related ({len(matches)/total*100:.1f}%)')
    return {"ai_reviews": matches, "total_scanned": total, "keyword_hits": len(matches)}


def embed_and_store(state: ReviewPipelineState) -> dict:
    """Embed AI-related reviews and store them in a local Chroma vector DB."""
    reviews = state["ai_reviews"]
    persist_dir = state["persist_dir"]
    batch_size = PIPELINE_CONFIG["embed_batch_size"]

    embeddings = HuggingFaceEmbeddings(
        model_name=PIPELINE_CONFIG["embedding_model"],
        model_kwargs={"device": "cpu"},
    )

    docs = [
        Document(
            page_content=str(r.get("message", "")),
            metadata={
                "key": str(r.get("key", "")),
                "score": int(float(r["score"])) if r.get("score") else 0,
                "timestamp": str(r.get("timestamp", "")),
                "language": str(r.get("language", "")),
                "app_version": str(r.get("app_version", "")),
                "thumbs_up_count": int(float(r["thumbs_up_count"])) if r.get("thumbs_up_count") else 0,
            }
        )
        for r in reviews
    ]

    vectorstore = None
    for i in tqdm(range(0, len(docs), batch_size), desc="Embedding & storing", unit="batch"):
        batch = docs[i : i + batch_size]
        if vectorstore is None:
            vectorstore = Chroma.from_documents(
                batch, embeddings, persist_directory=persist_dir
            )
        else:
            vectorstore.add_documents(batch)

    print(f'Stored {len(docs):,} documents in Chroma at {persist_dir!r}')
    return {"vectorstore_ready": True}


def summarize_run(state: ReviewPipelineState) -> dict:
    """Print a final run summary."""
    total = state["total_scanned"]
    hits = state["keyword_hits"]
    ready = state["vectorstore_ready"]
    persist = state["persist_dir"]
    pct = hits / total * 100 if total else 0
    summary = (
        f'Run complete.\n'
        f'  Rows scanned    : {total:>10,}\n'
        f'  AI reviews found: {hits:>10,}  ({pct:.2f}%)\n'
        f'  Vector DB ready : {ready}\n'
        f'  Persisted at    : {persist}'
    )
    print(summary)
    return {"run_summary": summary}


print("Pipeline 2 nodes defined.")

Pipeline 2 nodes defined.


In [20]:
# --- BUILD PIPELINE 2 GRAPH ---
review_builder = StateGraph(ReviewPipelineState)
review_builder.add_node("stream_and_filter", stream_and_filter)
review_builder.add_node("embed_and_store", embed_and_store)
review_builder.add_node("summarize_run", summarize_run)

review_builder.add_edge(START, "stream_and_filter")
review_builder.add_edge("stream_and_filter", "embed_and_store")
review_builder.add_edge("embed_and_store", "summarize_run")
review_builder.add_edge("summarize_run", END)

review_graph = review_builder.compile()
print("Review pipeline graph compiled.")

Review pipeline graph compiled.


In [21]:
# Run the full pipeline.
# TIP: For a quick smoke test, temporarily add  nrows=50_000  to the
#      pd.read_csv() call inside stream_and_filter before running the full dataset.
initial_state = {
    "csv_path": PIPELINE_CONFIG["csv_path"],
    "persist_dir": PIPELINE_CONFIG["chroma_persist_dir"],
    "ai_reviews": [],
    "total_scanned": 0,
    "keyword_hits": 0,
    "vectorstore_ready": False,
    "run_summary": "",
}

for event in review_graph.stream(initial_state, stream_mode="updates"):
    node_name = next(iter(event.keys()))
    print(f"-- Node completed: {node_name}")

print("Pipeline complete.")

































































































































































































































































































Scanning reviews: 662chunk [00:35, 18.61chunk/s]


Scanned 3,305,384 rows — 80,992 AI-related (2.5%)
-- Node completed: stream_and_filter


Exception ignored in: <function tqdm.__del__ at 0x000001F2A97F20C0>
Traceback (most recent call last):
  File "c:\Users\prash\anaconda3\envs\legalbot\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "c:\Users\prash\anaconda3\envs\legalbot\Lib\site-packages\tqdm\notebook.py", line 280, in close
    self.disp(bar_style='success', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
Embedding & storing: 100%|██████████| 17/17 [08:42<00:00, 30.73s/batch]

Stored 80,992 documents in Chroma at './chroma_reviews_db'
-- Node completed: embed_and_store
Run complete.
  Rows scanned    :  3,305,384
  AI reviews found:     80,992  (2.45%)
  Vector DB ready : True
  Persisted at    : ./chroma_reviews_db
-- Node completed: summarize_run
Pipeline complete.


In [ ]:
# --- SEARCH INTERFACE ---
# Safe to re-run: reconnects to the existing persisted Chroma DB without
# re-running the ingestion pipeline.
embeddings = HuggingFaceEmbeddings(model_name=PIPELINE_CONFIG["embedding_model"])
vectorstore = Chroma(
    persist_directory=PIPELINE_CONFIG["chroma_persist_dir"],
    embedding_function=embeddings,
)
print(f'Loaded vector store with {vectorstore._collection.count():,} documents.')


def search_ai_reviews(query: str, top_k: int = 10, filters: dict = None):
    """Semantic similarity search over AI-related app reviews.

    Args:
        query:   Natural language search query.
        top_k:   Number of results to return.
        filters: Optional Chroma metadata filter dict.
                 Examples: {"score": {"$gte": 4}}  or  {"language": "en"}
    """
    results = vectorstore.similarity_search_with_score(query, k=top_k, filter=filters)
    if not results:
        print("No results found.")
        return
    for doc, score in results:
        m = doc.metadata
        print(
            f'[sim:{score:.3f}]  Rating:{m.get("score")}/5  '
            f'Lang:{m.get("language")}  '
            f'Date:{str(m.get("timestamp", ""))[:10]}'
        )
        print(f'  {doc.page_content[:280]}')
        print()


# --- Example queries ---
print("=" * 60)
print("Query: AI autocomplete giving wrong suggestions")
print("=" * 60)
search_ai_reviews("AI autocomplete giving wrong suggestions")

print("=" * 60)
print("Query: love the new AI features  (positive reviews only)")
print("=" * 60)
search_ai_reviews("love the new AI features", filters={"score": {"$gte": 4}})